In [8]:
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 53.9 MB/s eta 0:00:00


# Sequence-Level Knowledge Distillation: Mistral-7B-Instruct -> SmolLM-360M on MATH-500

**What changed from the previous version, and why:**

1. **Logit-level KL distillation doesn't work here.** The teacher (Mistral-7B, vocab ~32k) and
   student (SmolLM-360M, vocab ~49k) use different tokenizers, so their logits live in different
   spaces — you cannot line them up token-for-token and take a KL divergence between them. That's
   why the original notebook's `DistillationTrainer` never trained anything usable.
2. **The grading logic was for GSM8K, not MATH-500.** MATH-500 answers are LaTeX expressions
   (e.g. `\frac{1}{2}`), not `#### 42`. This notebook extracts `\boxed{...}` content and grades
   with light normalization (and SymPy equivalence when available).
3. **This notebook uses *sequence-level* distillation instead:** the teacher generates full
   worked solutions for training problems, and the student is LoRA fine-tuned via ordinary
   supervised fine-tuning (causal LM loss) to imitate those teacher solutions. This works
   regardless of tokenizer mismatch, and is the standard technique for distilling between
   different model families.
4. **Runs on GPU with 4-bit quantization** instead of CPU — a 7B model on CPU is impractical
   for this notebook's runtime. Requires a Colab GPU runtime (T4 is enough).

**Before running:** `mistralai/Mistral-7B-Instruct-v0.2` is a gated model. Accept the license on
its Hugging Face page, and set an `HF_TOKEN` secret in Colab (or `huggingface-cli login`) with
read access.


## Setup

In [1]:
# Install necessary libraries
!pip install -q -U transformers datasets accelerate peft bitsandbytes sympy tqdm


In [2]:
import os
import re
import random
import torch
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type != "cuda":
    print("WARNING: no GPU detected. Loading Mistral-7B in 4-bit will be very slow or may fail "
          "on CPU. In Colab: Runtime -> Change runtime type -> GPU (T4).")


Using device: cuda


## Config

In [3]:
teacher_model_name = "mistralai/Mistral-7B-Instruct-v0.2"  # A slightly bigger model
student_model_name = "HuggingFaceTB/SmolLM-360M"

NUM_EVAL_PROBLEMS = 100   # problems used for baseline/final evaluation, per the task
NUM_TRAIN_PROBLEMS = 50   # subset of those 100 the teacher generates solutions for (training data)
MAX_NEW_TOKENS_SOLUTION = 400   # generation length for full worked solutions (teacher + eval)
ADAPTER_DIR = "distilled_student_adapter"

# Hugging Face auth (needed for the gated Mistral repo)
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if not os.environ.get("HF_TOKEN"):
    print("No HF_TOKEN found. If Mistral-7B-Instruct-v0.2 download fails with a 401/403, "
          "set an HF_TOKEN secret (Colab) or run `huggingface-cli login`.")


## Load Dataset

In [4]:
from datasets import load_dataset

print("Loading HuggingFaceH4/MATH-500 dataset...")
dataset = load_dataset("HuggingFaceH4/MATH-500")

# MATH-500 ships a single 'test' split; use the first NUM_EVAL_PROBLEMS problems throughout.
math_problems = dataset["test"].select(range(NUM_EVAL_PROBLEMS))

print(f"Loaded {len(math_problems)} problems for benchmarking.")
print("Example:")
print(math_problems[0])


Loading HuggingFaceH4/MATH-500 dataset...
Loaded 100 problems for benchmarking.
Example:
{'problem': 'Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\\theta),$ where $r > 0$ and $0 \\le \\theta < 2 \\pi.$', 'solution': 'We have that $r = \\sqrt{0^2 + 3^2} = 3.$  Also, if we draw the line connecting the origin and $(0,3),$ this line makes an angle of $\\frac{\\pi}{2}$ with the positive $x$-axis.\n\n[asy]\nunitsize(0.8 cm);\n\ndraw((-0.5,0)--(3.5,0));\ndraw((0,-0.5)--(0,3.5));\ndraw(arc((0,0),3,0,90),red,Arrow(6));\n\ndot((0,3), red);\nlabel("$(0,3)$", (0,3), W);\ndot((3,0), red);\n[/asy]\n\nTherefore, the polar coordinates are $\\boxed{\\left( 3, \\frac{\\pi}{2} \\right)}.$', 'answer': '\\left( 3, \\frac{\\pi}{2} \\right)', 'subject': 'Precalculus', 'level': 2, 'unique_id': 'test/precalculus/807.json'}


## Answer Extraction & Grading

MATH-500 answers are LaTeX expressions, so we pull the contents of the last `\boxed{...}` in the
generated text (falling back to the last `$...$` or number if no box is present), normalize
whitespace/formatting, and compare. When SymPy is available we also try a symbolic equivalence
check so e.g. `1/2` and `\frac{1}{2}` are treated as equal.

In [5]:
def extract_boxed(text: str):
    """Return the contents of the last \\boxed{...} in text, handling nested braces."""
    key = "\\boxed{"
    start = text.rfind(key)
    if start == -1:
        return None
    i = start + len(key)
    depth = 1
    out = []
    while i < len(text) and depth > 0:
        c = text[i]
        if c == "{":
            depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                break
        out.append(c)
        i += 1
    return "".join(out).strip()


def extract_predicted_answer(generated_text: str):
    boxed = extract_boxed(generated_text)
    if boxed:
        return boxed
    # Fallback: last inline-math segment
    dollar_matches = re.findall(r"\$(.+?)\$", generated_text)
    if dollar_matches:
        return dollar_matches[-1].strip()
    # Last-resort fallback: last number in the text
    num_matches = re.findall(r"-?\d+\.?\d*", generated_text)
    if num_matches:
        return num_matches[-1]
    return None


def normalize_answer(ans: str):
    if ans is None:
        return None
    a = ans.strip()
    a = a.replace("\\!", "").replace("\\,", "").replace(" ", "")
    a = a.replace("\\left", "").replace("\\right", "")
    a = a.strip("$")
    if a.endswith("."):
        a = a[:-1]
    return a


try:
    import sympy
    from sympy.parsing.latex import parse_latex

    def _sympy_equal(a: str, b: str):
        try:
            return bool(sympy.simplify(parse_latex(a) - parse_latex(b)) == 0)
        except Exception:
            return None
except Exception:
    def _sympy_equal(a: str, b: str):
        return None


def answers_match(predicted: str, true: str) -> bool:
    p, t = normalize_answer(predicted), normalize_answer(true)
    if p is None or t is None:
        return False
    if p == t:
        return True
    try:
        if float(p) == float(t):
            return True
    except (ValueError, TypeError):
        pass
    sym_result = _sympy_equal(p, t)
    if sym_result is not None:
        return sym_result
    return False


## Load Teacher Model (4-bit quantized)

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"Loading teacher model: {teacher_model_name}")
teacher_tokenizer = AutoTokenizer.from_pretrained(teacher_model_name, token=os.environ.get("HF_TOKEN"))
teacher_model = AutoModelForCausalLM.from_pretrained(
    teacher_model_name,
    quantization_config=bnb_config,
    device_map="auto",
    token=os.environ.get("HF_TOKEN"),
)
teacher_model.eval()
print("Teacher model loaded.")


Loading teacher model: mistralai/Mistral-7B-Instruct-v0.2


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Teacher model loaded.


## Load Student Model + LoRA Adapters

In [7]:
from peft import LoraConfig, get_peft_model

print(f"Loading student model: {student_model_name}")
student_tokenizer = AutoTokenizer.from_pretrained(student_model_name)
if student_tokenizer.pad_token is None:
    student_tokenizer.pad_token = student_tokenizer.eos_token

# Base student, kept unmodified — used for the "student baseline" eval below.
student_base_model = AutoModelForCausalLM.from_pretrained(student_model_name).to(device)
student_base_model.eval()

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

# Separate copy of the student that will actually be fine-tuned.
student_train_model = AutoModelForCausalLM.from_pretrained(student_model_name).to(device)
student_train_model = get_peft_model(student_train_model, lora_config)
student_train_model.print_trainable_parameters()


Loading student model: HuggingFaceTB/SmolLM-360M


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 819,200 || all params: 362,640,320 || trainable%: 0.2259


## Generation Helper

In [8]:
SOLUTION_INSTRUCTION = (
    "Solve the following math problem step by step. "
    "End your response with the final answer written as \\boxed{{answer}}.\n\n"
    "Problem: {problem}"
)


def build_prompt(tokenizer, problem: str) -> str:
    user_msg = SOLUTION_INSTRUCTION.format(problem=problem)
    if getattr(tokenizer, "chat_template", None):
        messages = [{"role": "user", "content": user_msg}]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    # Plain-text fallback for base (non-instruct) models like SmolLM-360M.
    return f"{user_msg}\nSolution:"


@torch.no_grad()
def generate_solution(model, tokenizer, problem: str, max_new_tokens=MAX_NEW_TOKENS_SOLUTION) -> str:
    prompt = build_prompt(tokenizer, problem)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


## Baseline Evaluation (Before Distillation)

In [9]:
def evaluate_model(model, tokenizer, problems, label="model"):
    correct = 0
    n = len(problems)
    for problem in tqdm(problems, desc=f"Evaluating {label}"):
        generated_text = generate_solution(model, tokenizer, problem["problem"])
        predicted = extract_predicted_answer(generated_text)
        if answers_match(predicted, problem["answer"]):
            correct += 1
    accuracy = 100.0 * correct / n
    print(f"{label}: {correct}/{n} correct ({accuracy:.2f}%)")
    return accuracy


In [10]:
print("\n--- Teacher Model Baseline ---")
teacher_baseline_accuracy = evaluate_model(teacher_model, teacher_tokenizer, math_problems, label="teacher (baseline)")



--- Teacher Model Baseline ---


Evaluating teacher (baseline):   0%|          | 0/100 [00:00<?, ?it/s]

teacher (baseline): 9/100 correct (9.00%)


In [11]:
print("\n--- Student Model Baseline ---")
student_baseline_accuracy = evaluate_model(student_base_model, student_tokenizer, math_problems, label="student (baseline)")



--- Student Model Baseline ---


Evaluating student (baseline):   0%|          | 0/100 [00:00<?, ?it/s]

student (baseline): 1/100 correct (1.00%)


## Build the Distillation Dataset

The teacher generates a full worked solution for each of the first `NUM_TRAIN_PROBLEMS` problems.
These (problem, teacher solution) pairs are the supervised targets the student will be fine-tuned
on — this is the "distillation" step, done at the sequence level rather than the logit level.

In [12]:
train_problems = math_problems.select(range(NUM_TRAIN_PROBLEMS))

teacher_solutions = []
for problem in tqdm(train_problems, desc="Generating teacher solutions"):
    solution_text = generate_solution(teacher_model, teacher_tokenizer, problem["problem"])
    teacher_solutions.append(solution_text)

print("Example teacher-generated solution:\n")
print(teacher_solutions[0])


Generating teacher solutions:   0%|          | 0/50 [00:00<?, ?it/s]

Example teacher-generated solution:

To convert a point from rectangular coordinates to polar coordinates, we use the following formulas:

x = r * cos(θ)
y = r * sin(θ)

Given the point (0, 3) in rectangular coordinates, we have:

x = 0
y = 3

Now we can find r and θ:

r = √(x² + y²) = √(0² + 3²) = √9 = 3

θ = arctan(y/x) = arctan(3/0) is undefined since x = 0.

However, we can use a special angle, π/2, to represent the vertical line x = 0 in polar coordinates. So, the point (0, 3) in rectangular coordinates corresponds to the point (3, π/2) in polar coordinates.

Therefore, the answer is:
\boxed{(3, \frac{\pi}{2})}


## Tokenize for Supervised Fine-Tuning

Each training example is `prompt + teacher_solution`. Prompt tokens are masked out with `-100` so
the loss only trains the student to produce the *solution*, not to reproduce the question.

In [13]:
MAX_SEQ_LEN = 512

def build_training_example(problem_text: str, solution_text: str):
    prompt = build_prompt(student_tokenizer, problem_text)
    full_text = prompt + solution_text + student_tokenizer.eos_token

    prompt_ids = student_tokenizer(prompt, add_special_tokens=False)["input_ids"]
    full = student_tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding="max_length",
        add_special_tokens=False,
    )

    labels = list(full["input_ids"])
    prompt_len = min(len(prompt_ids), len(labels))
    for i in range(prompt_len):
        labels[i] = -100
    # Also mask padding tokens.
    for i, tok in enumerate(full["input_ids"]):
        if tok == student_tokenizer.pad_token_id and i >= prompt_len:
            labels[i] = -100

    full["labels"] = labels
    return full


train_examples = [
    build_training_example(p["problem"], sol)
    for p, sol in zip(train_problems, teacher_solutions)
]

from datasets import Dataset
train_dataset = Dataset.from_list(train_examples)
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
print(f"Distillation training dataset size: {len(train_dataset)}")


Distillation training dataset size: 50


## Fine-Tune the Student (LoRA) on Teacher Solutions

In [14]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./distillation_results",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="no",
    report_to=[],
)

trainer = Trainer(
    model=student_train_model,
    args=training_args,
    train_dataset=train_dataset,
)

trainer.train()

student_train_model.save_pretrained(ADAPTER_DIR)
student_tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Saved LoRA adapter to ./{ADAPTER_DIR}")


Step,Training Loss
5,0.826998
10,0.839525
15,0.803060
20,0.787471


Saved LoRA adapter to ./distilled_student_adapter


## Reload the Distilled Student

Load a fresh copy of the base student model and attach the LoRA adapter we just trained (this is
the correct way to reload a PEFT-trained model — the original notebook tried to `from_pretrained`
an adapter directory as if it were a full model, which is why it failed with a 404).

In [15]:
from peft import PeftModel

distilled_base = AutoModelForCausalLM.from_pretrained(student_model_name).to(device)
distilled_student_model = PeftModel.from_pretrained(distilled_base, ADAPTER_DIR)
distilled_student_model.eval()
print("Distilled student model loaded.")


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Distilled student model loaded.


## Evaluation After Distillation

In [16]:
print("\n--- Distilled Student Model Evaluation ---")
distilled_student_accuracy = evaluate_model(
    distilled_student_model, student_tokenizer, math_problems, label="distilled student"
)



--- Distilled Student Model Evaluation ---


Evaluating distilled student:   0%|          | 0/100 [00:00<?, ?it/s]

distilled student: 1/100 correct (1.00%)


## Summary of Results

In [17]:
print(f"Teacher Model Baseline Accuracy:      {teacher_baseline_accuracy:.2f}%")
print(f"Student Model Baseline Accuracy:      {student_baseline_accuracy:.2f}%")
print(f"Distilled Student Model Accuracy:     {distilled_student_accuracy:.2f}%")

if distilled_student_accuracy > student_baseline_accuracy:
    print("\nConclusion: Distillation improved the student model's accuracy on MATH-500.")
elif distilled_student_accuracy < student_baseline_accuracy:
    print("\nConclusion: Distillation did not improve accuracy (or it decreased). "
          "Consider more training problems, more epochs, or a larger LoRA rank.")
else:
    print("\nConclusion: Distillation did not change accuracy.")


Teacher Model Baseline Accuracy:      9.00%
Student Model Baseline Accuracy:      1.00%
Distilled Student Model Accuracy:     1.00%

Conclusion: Distillation did not change accuracy.


## Notes & Limitations

- **Answer grading is still approximate.** MATH-500 answers can be written in many equivalent
  forms; the SymPy-based check catches common cases (fractions, simple algebra) but won't catch
  everything. For rigorous grading, consider the `math-verify` or `minerva_math` evaluators used
  in published MATH benchmarks.
- **50 training examples is a small demonstration set.** Real distillation runs typically use
  thousands of teacher-generated examples. Increase `NUM_TRAIN_PROBLEMS` (you can also draw from
  the full MATH training set rather than the 100 held out for eval) and epochs for a stronger effect.
- **Greedy decoding (`do_sample=False`)** is used for reproducibility; sampling with multiple
  teacher completions per problem (and keeping only the correct ones) usually produces a cleaner
  distillation dataset.
